In [ ]:
from sklearn.linear_model import LinearRegression
import pickle
import numpy as np
from flask import Flask, request, render_template

# STEP 1: Train and Save Model
X = np.array([[1], [2], [3], [4], [5]])
y = np.array([2, 4, 6, 8, 10])

model = LinearRegression()
model.fit(X, y)

with open("model.pkl", "wb") as file:
    pickle.dump(model, file)
print("Model saved successfully")

In [ ]:
import os

# STEP 3: Create HTML File (This would typically be in 'templates/index.html')
# For demonstration, this is shown here as a string, but in a real Flask app,
# it would be a separate file.
html_content = """
<!DOCTYPE html>
<html>
<head>
    <title>ML Model Deployment</title>
</head>
<body>
    <h2>Linear Regression Model</h2>
    <form action="/predict" method="post">
        Enter value:
        <input type="text" name="value" required>
        <button type="submit">Predict</button>
    </form>
    <h3>{{ result }}</h3>
</body>
</html>
"""

# Create the 'templates' directory if it doesn't exist
if not os.path.exists('templates'):
    os.makedirs('templates')

# Save the HTML content to index.html inside the templates folder
with open('templates/index.html', 'w') as f:
    f.write(html_content)

print('templates/index.html created successfully.')

In [ ]:
# Install flask-ngrok and pyngrok
!pip install flask-ngrok pyngrok -qq

In [ ]:
from flask import Flask, request, render_template
import pickle
from pyngrok import ngrok, conf
from google.colab import userdata # Import userdata for secrets

# STEP 2: Create Flask Application
app = Flask(__name__)

# Load trained model
model = pickle.load(open("model.pkl", "rb"))

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/predict", methods=["POST"])
def predict():
    value = float(request.form["value"])
    prediction = model.predict([[value]])
    return render_template("index.html", result=f"Predicted Output: {prediction[0]}")


print("Starting ngrok tunnel...")
ngrok.kill() # Close any existing tunnels
tunnel = ngrok.connect(8000) # Connect ngrok to Flask's default port, changed from 5000 to 8000
public_url = tunnel.public_url
print(f"* Ngrok Tunnel URL: {public_url}")
print("You can access your Flask app at this URL.")

# Run Flask app (blocking operation, must be the last line in the cell)
app.run(port=8000, use_reloader=False) # Changed port from 5000 to 8000
